# P26 — Control a nivel humano mediante aprendizaje por refuerzo profundo

## 1. Título y paper

**Paper:** *Human-level control through deep reinforcement learning*  
**Autoría:** Volodymyr Mnih, Koray Kavukcuoglu, David Silver, y otros (DeepMind)  
**Año y venue:** 2015 · Nature 518, 529–533 (2015)  
**Nivel:** L3 · **Motor:** `dqn`  
**Ficha completa:** [`P26_dqn`](../../papers/foundational/P26_dqn/README.md)

**Hito:** El primer agente que aprende a actuar directamente desde píxeles, con la misma arquitectura y los mismos hiperparámetros en decenas de juegos.

- [DOI (Nature 518, 529–533)](https://doi.org/10.1038/nature14236)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Combinar aprendizaje por refuerzo con aproximación de función no lineal era notoriamente inestable: las muestras consecutivas están correlacionadas y el objetivo se mueve mientras se aprende.
2. Ejecutar una implementación mínima de la propuesta: Q-learning con una red convolucional, estabilizado con repetición de experiencia (rompe la correlación) y una red objetivo congelada (fija el blanco).
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P02
- P04
- Watkins (1989), Q-learning


## 4. Intuición

Aprender a jugar sin que nadie te explique las reglas: pruebas, ves el marcador, y ajustas. El problema es que si aprendes solo de lo último que acabas de hacer, te obsesionas con ello y olvidas lo demás.


## 5. Concepto mínimo

```text
Q-learning:   Q(s,a) ← Q(s,a) + α·[ r + γ·max_a' Q(s',a') − Q(s,a) ]

Dos estabilizaciones que aporta el paper:
  · repetición de experiencia: guardar (s,a,r,s') y muestrear un LOTE al azar
      → rompe la correlación entre muestras consecutivas
  · red objetivo: usar una copia CONGELADA de Q para calcular el objetivo
      → el blanco deja de moverse mientras se dispara
```


## 6. Código explicado

El motor entrena Q tabular en una rejilla 4×4, con y sin las dos estabilizaciones.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('dqn', seed=7)['result']
print('entorno:', r['entorno'], '\n')
show(r['con_replay_y_red_objetivo'])
show(r['sin_replay_ni_red_objetivo'])

## 7. Predicción antes de ejecutar

1. ¿Cuál es el número mínimo de pasos de (0,0) a (3,3) moviéndose en cruz?
2. ¿Cuál de las dos configuraciones se acercará más a ese óptimo?
3. ¿Por qué aprender de transiciones consecutivas es un problema?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('dqn', seed=semilla)['result']
    con = r['con_replay_y_red_objetivo']['pasos_medios_ultimos_50']
    sin = r['sin_replay_ni_red_objetivo']['pasos_medios_ultimos_50']
    print(f'semilla {semilla:>2} · con estabilizaciones {con:.2f} · sin ellas {sin:.2f} '
          f'· óptimo 6')

## 9. Salida interpretable

Con las dos estabilizaciones la política converge cerca del óptimo. Sin ellas queda peor y con más varianza entre semillas. **La contribución del paper no es Q-learning** —que es de 1989— sino hacerlo estable cuando la función Q es una red neuronal.


## 10. Comentario pedagógico

Fíjate en lo que el paper NO cambia: la misma arquitectura y los mismos hiperparámetros en decenas de juegos distintos. Esa uniformidad es la afirmación fuerte —generalidad— y es más difícil de conseguir que un buen resultado en un juego concreto.


## 11. Error o anti-patrón deliberado

Anti-patrón: mover el objetivo mientras se persigue. Sin red congelada, cada actualización cambia el blanco de la siguiente.


In [ ]:
Q = 0.0
print('sin red objetivo — el blanco se mueve con cada paso:')
for i in range(5):
    objetivo = 1.0 + 0.95 * Q      # el objetivo depende de la MISMA Q que se actualiza
    Q += 0.5 * (objetivo - Q)
    print(f'  paso {i}: objetivo={objetivo:.4f}  Q={Q:.4f}')

## 12. Corrección

Con una copia congelada, el blanco se queda quieto entre sincronizaciones:


In [ ]:
Q, Q_obj = 0.0, 0.0
print('con red objetivo — el blanco solo cambia al sincronizar:')
for i in range(5):
    objetivo = 1.0 + 0.95 * Q_obj
    Q += 0.5 * (objetivo - Q)
    if i == 2:
        Q_obj = Q
        print('  --- sincronización ---')
    print(f'  paso {i}: objetivo={objetivo:.4f}  Q={Q:.4f}')

## 13. Desafío guiado

Sube la tasa de exploración ε y observa el compromiso entre explorar y explotar.


In [ ]:
for eps in (0.0, 0.05, 0.2, 0.6):
    print(f'ε={eps:<5} → ' + ('nunca descubre rutas nuevas' if eps == 0 else
          'explora demasiado, no explota lo aprendido' if eps > 0.5 else 'equilibrio razonable'))

## 14. Desafío autónomo

Implementa DQN con una red pequeña sobre un entorno de control clásico y de código abierto. Mide la curva de recompensa con y sin repetición de experiencia, con tres semillas, y reporta la varianza además de la media.


## 15. Evidencia de aprendizaje

Guarda la comparación con y sin estabilizaciones en tres semillas, y tu explicación de por qué un objetivo móvil desestabiliza el aprendizaje.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P26_dqn/README.md) · evaluación formal: [`assessments/papers/P26_dqn.md`](../../assessments/papers/P26_dqn.md)


## 16. Cierre

Ya hay un agente que aprende a decidir por recompensa. Pero en juegos con un espacio enorme, probar no basta: hay que **buscar**.


## 17. Conexión con el siguiente hito

- P27
- P12

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
